In [12]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [13]:
load_dotenv(override=True)
openai = OpenAI()

In [14]:
# Load Markdown file content into the variable `data`
with open("aquarium.md", "r", encoding="utf-8") as file:
    data = file.read()

# Optional quick check
print(f"Loaded {len(data)} characters from aquariam.md")

Loaded 5694 characters from aquariam.md


In [15]:
system_prompt = f"""

# Your role

You are an AI assistant representing Mode Aquarium.
Your job is to answer questions from customers about the aquarium — its services,
offerings, and information — in a helpful, friendly, and professional way.
If asked, you explain clearly that you are an AI assistant for Mode Aquarium.

# Knowledge

You must answer ONLY using the knowledge provided below. This is your single source
of truth. Do not use any outside information.

{data}

# Rules

- Answer customer questions using ONLY the knowledge provided above.
- NEVER make assumptions, guess, or make up an answer.
- If a question cannot be answered from the provided knowledge, simply say that you
  don't have that information and offer to help with something else.
- Do NOT answer anything that is unrelated to Mode Aquarium; politely steer the
  conversation back to how you can help with the aquarium.
- Always be helpful, warm, and professional.

IMPORTANT: Strictly do not answer anything beyond the knowledge passed to you.
If you don't know, say so — never invent an answer.
"""

In [16]:
user_prompt=f"""
 what is your operating time?
 """

In [21]:
message= [
    {"role":"system","content":system_prompt},
    {"role":"user","content":user_prompt},
]

In [22]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=message)
display(Markdown(response.choices[0].message.content))

I don’t have information about Mode Aquarium’s operating hours.

If you’d like, I can still help with product details, custom aquarium quotes, shipping, installation, or warranty questions.

In [23]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [24]:
record_email_tool("test@testy.com")

Tool called to record an email: test@testy.com


'Email received'

In [25]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [26]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [27]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
# ---- Evaluator LLM ----
# A second model that checks the main assistant's reply BEFORE it reaches the customer.
# It uses the same knowledge as ground truth and decides if the answer is acceptable.

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


evaluator_system_prompt = f"""
You are a strict quality evaluator for an AI assistant that represents Mode Aquarium.
Your job is to judge whether the assistant's latest reply to a customer is acceptable
to send.

You are given the same knowledge base that the assistant must rely on. Treat it as the
ONLY source of truth.

# Knowledge base

{data}

# What makes a reply ACCEPTABLE

- It is fully supported by the knowledge base (no invented facts, prices, times, etc.).
- If the answer is not in the knowledge base, the assistant honestly says it does not
  have that information instead of guessing.
- It stays on the topic of Mode Aquarium.
- It is clear, helpful, and professional — not vague or evasive when the info IS available.

# What makes a reply UNACCEPTABLE

- It states anything not backed by the knowledge base (hallucination / made-up info).
- It makes assumptions or guesses.
- It is vague or unhelpful even though the knowledge base contains the answer.
- It answers off-topic questions instead of steering back to the aquarium.

Reply with is_acceptable (true/false) and concrete feedback the assistant can use to fix
the reply if it is not acceptable.
"""


def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here is the conversation history:\n\n{history}\n\n"
    user_prompt += f"Here is the latest message from the customer:\n\n{message}\n\n"
    user_prompt += f"Here is the assistant's proposed reply:\n\n{reply}\n\n"
    user_prompt += "Evaluate whether this reply is acceptable to send, and give feedback."
    return user_prompt


def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]
    response = openai.beta.chat.completions.parse(
        model="gpt-5.4-mini", messages=messages, response_format=Evaluation
    )
    return response.choices[0].message.parsed

In [ ]:

def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt
    updated_system_prompt += "\n\n# Previous attempt was rejected by the quality checker\n"
    updated_system_prompt += f"Your previous reply was:\n{reply}\n\n"
    updated_system_prompt += f"It was rejected for this reason:\n{feedback}\n\n"
    updated_system_prompt += "Please generate a new reply that fixes this. Stay strictly within the knowledge base."

    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:

MAX_RETRIES = 2

def generate_reply(message, history):
    """Run the main assistant, handling any tool calls, and return the final text."""
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        messages.append(tool_message)
        for tool_call in tool_message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    return response.choices[0].message.content


def chat(message, history):
    reply = generate_reply(message, history)

    for attempt in range(MAX_RETRIES):
        evaluation = evaluate(reply, message, history)
        if evaluation.is_acceptable:
            print(f"Evaluator: PASSED (attempt {attempt + 1})")
            break
        print(f"Evaluator: REJECTED (attempt {attempt + 1}) -> {evaluation.feedback}")
        reply = rerun(reply, message, history, evaluation.feedback)

    return reply

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Evaluator: PASSED (attempt 1)


d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Evaluator: PASSED (attempt 1)


...

d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Evaluator: PASSED (attempt 1)


d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: msirajkazigmail.com
Evaluator: PASSED (attempt 1)


d:\Agentic&MCP\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
